# Optimizer comparison: reproducible MLP traces / 优化器比较：可复现 MLP 轨迹
This bilingual NumPy notebook replays two clearly separated comparisons from the shared TypeScript engine. / 本双语 NumPy Notebook 从共享 TypeScript 引擎重放两组清晰区分的比较。

- **First-step norm matched / 首步范数匹配：** all optimizers receive learning rates derived from the same initial full-batch gradient, so their first full-vector update norms agree. / 所有优化器的学习率都由同一个初始全批梯度推导，因此首个完整向量更新范数一致。
- **Predeclared practical / 预先声明的实用设置：** independently chosen practical learning rates are retained as a separate result and must not be compared as if they were norm matched. / 独立选择的实用学习率作为单独结果保留，不能与范数匹配结果混为一谈。
- The fixed Banknote split fits standardization on training rows only. / 固定的 Banknote 划分仅用训练行拟合标准化。


In [1]:
import json
from pathlib import Path
import numpy as np

def resolve_asset_dir():
    for root in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (root, root / "public" / "notebooks" / "optimizer-comparison"):
            if (candidate / "optimizer-comparison-trajectories.json").is_file():
                return candidate
    raise FileNotFoundError("Could not find optimizer-comparison trajectories from the kernel working directory")

asset_dir = resolve_asset_dir()
payload = json.loads((asset_dir / "optimizer-comparison-trajectories.json").read_text(encoding="utf-8"))
rows = payload["rows"]
assert len(rows) == 328
matched = [row for row in rows if row["comparison"] == "first-step-norm-matched" and row["update"] == 1]
practical = [row for row in rows if row["comparison"] == "predeclared-practical" and row["update"] == 1]
matched_norms = np.array([row["updateNorm"] for row in matched], dtype=float)
assert len(matched) == 4 and np.ptp(matched_norms) < 1e-10
assert len(practical) == 4
print(f"Replayed {len(rows)} rows from {asset_dir.name}; matched first-step norm = {matched_norms[0]:.12f}")


Replayed 328 rows from optimizer-comparison; matched first-step norm = 0.002141565511


In [2]:
banknote_path = asset_dir.parents[1] / "datasets" / "optimizer-comparison" / "banknote-transfer.json"
banknote = json.loads(banknote_path.read_text(encoding="utf-8"))
assert banknote["preprocessing"]["fitSplit"] == "train"
assert banknote["splitCounts"] == {"train": 960, "validation": 206, "test": 206}
print("Banknote replay uses the fixed split and train-only standardization.")


Banknote replay uses the fixed split and train-only standardization.
